# Step 7: A closer look at the chosen model

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** Step 6 (`chosen.json`). **Saves:** `docs/img/step7_test_week.png`.

How the chosen model's test errors compare with the baseline overall and zone by zone, and what its predictions look
like for the busiest zone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
from statsmodels.tools.eval_measures import meanabs, rmse

chosen = mu.read("chosen", made_by="step6_compare.ipynb")
res_final = mu.ols(chosen["terms"])
print(f"chosen model: {chosen['name']}")
test["model"] = np.expm1(res_final.predict(test)).clip(lower=0)
print("holidays in the test days:", ", ".join(test.loc[test["holiday"] == 1, "date"].drop_duplicates().dt.strftime("%b %d")))

summary = pd.DataFrame({
    "RMSE": [rmse(test["model"], test["pickups"]), rmse(test["lag168"], test["pickups"])],
    "MAE": [meanabs(test["model"], test["pickups"]), meanabs(test["lag168"], test["pickups"])],
}, index=[f"chosen model ({chosen['name']})", "baseline: same hour last week"])
by_zone = test.groupby("zone").apply(lambda g: pd.Series({
    "mean pickups": g["pickups"].mean(), "RMSE model": rmse(g["model"], g["pickups"]),
    "RMSE baseline": rmse(g["lag168"], g["pickups"])}), include_groups=False)
by_zone["model wins"] = by_zone["RMSE model"] < by_zone["RMSE baseline"]
print(f"model beats the baseline in {by_zone['model wins'].sum()} of the {len(by_zone)} zones")
summary

In [ ]:
by_zone.sort_values("mean pickups", ascending=False).head(15)

In [ ]:
zone = by_zone["mean pickups"].idxmax()
week = test[(test["zone"] == zone) & (test["hour"].dt.month == 6)].set_index("hour")
week = week.reindex(pd.date_range("2025-06-01", "2025-06-30 23:00", freq="h"))  # gaps = training days
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(week.index, week["pickups"], color=mu.BLUE, label="actual", lw=2)
ax.plot(week.index, week["model"], color=mu.ORANGE, label="chosen model", lw=1.6)
ax.plot(week.index, week["lag168"], color=mu.AQUA, label="same hour last week", lw=1.6)
ax.set(title=f"Zone {zone}, June test days (gaps are training days): hourly pickups", ylabel="pickups per hour")
ax.legend(frameon=False, ncols=3, loc="upper left")
fig.tight_layout()
mu.save_figure(fig, "step7_test_week.png")
plt.show()

**Results.**
- **Overall:** on the test days, the full model has RMSE **34.1** and MAE **19.2** pickups per zone-hour. The
  baseline has 40.1 and 22.0, so the model's typical error is **15% smaller**.
- **By zone:** it beats the baseline in **49 of the 50 zones**.
- **Holidays:** three holidays fell in the random test days (Feb 17, Jul 4, Nov 27).
- **Zone 237 (Upper East Side South), June test days:** the model follows the daily cycle closely. On June 23 it
  under-predicts the midday peak, which was unusually high that day.

## Notes for the report

- **One model for both questions.** The full model explains *why* demand changes (rain +2.2% per mm, etc.) and also
  forecasts best on the test days.
- **The model assumes it is used one day ahead,** since it needs yesterday's demand.
- **Demand = completed pickups.** People who couldn't find a taxi aren't counted, so busy hours are probably
  underestimated.